# TrainWise — Acute:Chronic Workload Ratio & EWMA Trend Analysis

**Python ML course project · Training-load monitoring for injury prevention**

This notebook is the data-science write-up behind the **Load Trend** charts in the TrainWise app
(the trainee *Load* tab and the coach *Analytics* screen). It re-implements, explains, evaluates and
visualises the two ways the app measures training load, and trains small models on top of them.

## Background

Athletes get injured when they ramp training **too fast** relative to what they are used to. The
sports-science tool for this is the **Acute : Chronic Workload Ratio (ACWR)** (Gabbett, 2016):

$$\text{ACWR} = \frac{\text{acute load (last 7 days)}}{\text{chronic load (last 28 days, weekly average)}}$$

- **ACWR < 0.8** — detraining / under-loaded
- **0.8 ≤ ACWR ≤ 1.3** — the *sweet spot* (fit and adapting, low injury risk)
- **ACWR > 1.3** — spiking load, elevated injury risk (**> 1.5** = danger)

TrainWise computes ACWR two ways, which this notebook builds and compares:

1. **Classic rolling** (Gabbett 2016) — simple moving sums. The official status the app and coach use.
2. **Smooth EWMA** (Williams 2017) — an exponentially-weighted moving average that weights recent
   sessions more, so it reacts faster to spikes and decays gradually after rest.

**Session load** in TrainWise is `duration (min) × exertion (RPE 1-10)` — e.g. a 45-min session at
RPE 6 = 270 load units. Everything below works from that per-session load.

## What this notebook does (mapping to the course tasks)

| Course task | Here |
|---|---|
| Data cleaning + EDA | §2 — synthetic-but-realistic training histories, distributions, correlations |
| Feature engineering | §3-§5 — daily load series, rolling ACWR, EWMA ACWR, monotony/strain |
| **Regression** (Task 1) | §7 — predict next-week load (MAE / MSE / RMSE, residuals) |
| **Classification** (Task 2) | §8 — predict an injury in the next 7 days (Accuracy / Precision / Recall / F1 / ROC-AUC) |
| **Clustering** | §9 — KMeans athlete archetypes |
| Model export | §10 — `joblib` pickle, the same pattern the live service loads |

> The **live** service (`ml/features.py`) reads these exact formulas from the real `ActivityLogs`
> table; here we generate reproducible synthetic data so the notebook runs standalone for grading.
> The formulas are byte-for-byte the same as `ml/features.py` and `utils/loadSeries.js` (verified by a
> parity test), so the numbers here match what the app shows.


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score,
)
import joblib

sns.set_theme(style="whitegrid", context="notebook")
RNG = np.random.default_rng(42)      # reproducible
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
print("Environment ready.")

## 2. Data — synthetic training histories

We simulate **12 athletes** over **140 days**. Each athlete has an experience level (Beginner /
Regular / Advanced) that sets their baseline volume, a weekly structure with rest days, and a macro
shape: an initial ramp, a steady block with an overreach spike, a taper, then a rebuild. Injuries
are seeded to follow high-load days (which is exactly the pattern ACWR is meant to catch).

The columns mirror the app's `ActivityLogs`: `date`, `load` (= duration × RPE), `duration`, `rpe`,
plus an `injury` flag (1 on the day an injury started).

In [ ]:
N_ATHLETES = 12
N_DAYS = 140
START = pd.Timestamp("2026-01-01")

EXPERIENCE = {1: "Beginner", 2: "Regular", 3: "Advanced"}
BOOTSTRAP = {1: 150.0, 2: 280.0, 3: 420.0}   # experience -> expected weekly acute (LoadParameters seed)
SESSIONS_PER_WEEK = {1: 3, 2: 4, 3: 5}

def synth_athlete(aid, exp):
    loads, durations, rpes, injuries = [], [], [], []
    cooldown = 0
    recent = []                    # trailing daily loads (acute proxy)
    chronic = BOOTSTRAP[exp]       # slow-moving weekly-equivalent load
    for d in range(N_DAYS):
        # macro phase multiplier: ramp -> block+spike -> taper -> rebuild
        if d < 21:      phase = 0.5 + d / 42.0
        elif d < 85:    phase = 1.0 + 0.35 * np.sin((d - 21) / 14.0)
        elif d < 105:   phase = 0.55
        else:           phase = 1.15
        train = (RNG.random() < SESSIONS_PER_WEEK[exp] / 7.0) and cooldown == 0
        if cooldown > 0:
            cooldown -= 1
        if not train:
            loads.append(0); durations.append(0); rpes.append(0); injuries.append(0)
            recent = (recent + [0.0])[-7:]
            chronic = 0.96 * chronic + 0.04 * 0.0
            continue
        dur = int(np.clip(RNG.normal(45, 15), 15, 120))
        rpe = int(np.clip(RNG.normal(5.5 * phase, 1.6), 1, 10))
        load = int(dur * rpe)                       # session load = duration x exertion
        recent = (recent + [float(load)])[-7:]
        acute7 = sum(recent)
        # Injury risk RISES with the acute:chronic spike (this is exactly what ACWR
        # is designed to catch), so the classifier later has a real signal to learn.
        ratio = acute7 / chronic if chronic > 0 else 0.0
        inj = 0
        p = 0.015 + max(0.0, ratio - 1.15) * 0.45
        if RNG.random() < p:
            inj = 1
            cooldown = int(RNG.integers(4, 12))     # forced easy/rest days after
        chronic = 0.96 * chronic + 0.04 * (load * 7.0)   # chronic tracks weekly-scaled load
        loads.append(load); durations.append(dur); rpes.append(rpe); injuries.append(inj)
    return pd.DataFrame({
        "athlete": aid,
        "experience": exp,
        "date": pd.date_range(START, periods=N_DAYS, freq="D"),
        "load": loads, "duration": durations, "rpe": rpes, "injury": injuries,
    })

frames = []
for aid in range(1, N_ATHLETES + 1):
    frames.append(synth_athlete(aid, int(RNG.integers(1, 4))))
df = pd.concat(frames, ignore_index=True)
print(df.shape)
df.head()

### 2.1 Cleaning & sanity checks

In [ ]:
# In the real DB we would drop unconfirmed rows and coerce nulls; here we just
# verify the synthetic frame is clean and well-formed.
print("Missing values:\n", df.isna().sum().to_string())
print("\nInjury episodes:", int(df["injury"].sum()))
print("Rest days (load == 0):", int((df["load"] == 0).sum()), "of", len(df))
df.describe()[["load", "duration", "rpe"]]

### 2.2 Exploratory plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df.loc[df["load"] > 0, "load"], bins=30, kde=True, ax=axes[0], color="#2a9d8f")
axes[0].set_title("Distribution of session load (training days)")
axes[0].set_xlabel("load = duration x RPE")

sns.boxplot(data=df[df["load"] > 0], x="experience", y="load", ax=axes[1],
            palette="viridis")
axes[1].set_xticklabels([EXPERIENCE[int(t.get_text())] for t in axes[1].get_xticklabels()])
axes[1].set_title("Session load by experience level")
plt.tight_layout(); plt.show()

In [ ]:
# One athlete's daily load over time — the raw signal ACWR summarises.
a = df[df["athlete"] == 1]
plt.figure(figsize=(13, 3.5))
plt.bar(a["date"], a["load"], width=1.0, color="#457b9d")
inj_days = a[a["injury"] == 1]
plt.scatter(inj_days["date"], inj_days["load"] + 40, marker="v", color="crimson",
            s=60, label="injury onset", zorder=5)
plt.title(f"Athlete 1 daily load  ({EXPERIENCE[int(a['experience'].iloc[0])]})")
plt.ylabel("session load"); plt.legend(); plt.tight_layout(); plt.show()

## 3. Feature engineering — daily load series

Multiple sessions on one day sum together; rest days are zero-filled so the rolling windows are
calendar-correct. This is exactly `features.daily_load_series` / `loadSeries.js`.

In [ ]:
def daily_series(g):
    s = g.groupby("date")["load"].sum().astype(float)
    idx = pd.date_range(g["date"].min(), g["date"].max(), freq="D")
    return s.reindex(idx, fill_value=0.0)

daily_by_athlete = {aid: daily_series(df[df["athlete"] == aid]) for aid in df["athlete"].unique()}
daily_by_athlete[1].head(10)

## 4. Classic rolling ACWR (Gabbett 2016)

- **acute** = sum of session loads over the trailing **7** days
- **chronic** = trailing **28**-day load, on a weekly-equivalent scale

A naïve `chronic = sum28 / 4` breaks for new athletes (a first big session divided by a near-zero
chronic reads an impossible ratio). TrainWise applies two guards, mirrored here exactly:

1. **Cold-start floor** — with **< 7 active days** in the trailing 28, floor chronic at the
   experience bootstrap (150 / 280 / 420 weekly). A brand-new athlete is judged against their level,
   not flagged at ratio 4.
2. **Covered-days ramp** — once ≥ 7 active days, divide by the weeks *actually covered*
   (`sum28 / min(4, covered/7)`), so a steady 2-week-old athlete reads ~1.0 instead of a false 2.0.
   Full 28-day histories are unchanged (covered = 28 → /4).

In [ ]:
ACUTE_W, CHRONIC_W, DIVISOR = 7, 28, 4.0

def rolling_acwr(daily, exp):
    acute = daily.rolling(ACUTE_W, min_periods=1).sum()
    sum28 = daily.rolling(CHRONIC_W, min_periods=1).sum()
    active = (daily > 0).rolling(CHRONIC_W, min_periods=1).sum().to_numpy()
    boot = BOOTSTRAP[exp]

    vals = daily.to_numpy(); n = len(vals); s28 = sum28.to_numpy()
    floored = np.maximum(s28 / DIVISOR, boot)                 # guard 1: cold-start floor
    chronic = floored.copy()
    nz = np.flatnonzero(vals > 0)
    if nz.size:                                               # guard 2: covered-days ramp
        idxs = np.arange(n)
        j = np.searchsorted(nz, idxs - (CHRONIC_W - 1), side="left")
        first = nz[np.minimum(j, nz.size - 1)]
        covered = np.maximum(idxs - first + 1, 7)
        ramped = s28 / np.minimum(DIVISOR, covered / 7.0)
        chronic = np.where(active >= 7, ramped, floored)
    ratio = np.where(chronic > 0, acute.to_numpy() / chronic, np.nan)
    return pd.DataFrame({"acute": acute, "chronic": chronic, "acwr_rolling": ratio},
                        index=daily.index)

roll1 = rolling_acwr(daily_by_athlete[1], int(df[df.athlete == 1]["experience"].iloc[0]))
roll1.tail()

## 5. Smooth EWMA ACWR (Williams 2017)

Instead of a flat window, weight each day exponentially with $\lambda = \frac{2}{N+1}$
(acute $N=7 \Rightarrow \lambda=0.25$; chronic $N=28 \Rightarrow \lambda\approx0.069$):

$$\text{EWMA}_t = \text{load}_t \cdot \lambda + (1-\lambda)\cdot \text{EWMA}_{t-1}$$

A zero-seeded EWMA under-estimates the average early on (which *inflates* the ratio — a first
workout would falsely read Red). We remove that zero-initialisation bias with the **Adam
correction** (Kingma & Ba, 2015), dividing by $1-(1-\lambda)^t$ where $t$ counts days from the first
logged session. This is the single subtlety that makes EWMA usable from day one.

In [ ]:
LAMBDA_A, LAMBDA_C = 2 / (ACUTE_W + 1), 2 / (CHRONIC_W + 1)   # 0.25, ~0.069

def ewma_acwr(daily, exp):
    boot_daily = BOOTSTRAP[exp] / 7.0
    vals = daily.to_numpy(); idx = daily.index; n = len(vals)
    active = (pd.Series(vals > 0, index=idx)
              .rolling(CHRONIC_W, min_periods=1).sum().to_numpy())
    nz = np.flatnonzero(vals > 0); first = int(nz[0]) if nz.size else None

    a = c = 0.0; t = 0; out = np.full(n, np.nan)
    for i in range(n):
        if first is not None and i >= first:
            t += 1
            a = vals[i] * LAMBDA_A + (1 - LAMBDA_A) * a
            c = vals[i] * LAMBDA_C + (1 - LAMBDA_C) * c
        if t > 0:
            a_corr = a / (1 - (1 - LAMBDA_A) ** t)            # Adam bias correction
            c_corr = c / (1 - (1 - LAMBDA_C) ** t)
            eff_c = c_corr if active[i] >= 7 else max(c_corr, boot_daily)
            out[i] = a_corr / eff_c if eff_c > 0 else np.nan
    return pd.Series(out, index=idx, name="acwr_ewma")

ewma1 = ewma_acwr(daily_by_athlete[1], int(df[df.athlete == 1]["experience"].iloc[0]))
ewma1.tail()

## 6. Rolling vs EWMA — how they compare

In [ ]:
aid = 1
exp = int(df[df.athlete == aid]["experience"].iloc[0])
daily = daily_by_athlete[aid]
roll = rolling_acwr(daily, exp)
ewma = ewma_acwr(daily, exp)

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.axhspan(0.8, 1.3, color="#2a9d8f", alpha=0.12, label="sweet spot 0.8-1.3")
ax.axhline(1.3, color="#e9c46a", ls="--", lw=1)
ax.axhline(1.5, color="#e76f51", ls="--", lw=1, label="danger 1.5")
ax.plot(roll.index, roll["acwr_rolling"], label="Classic rolling", color="#264653", lw=2)
ax.plot(ewma.index, ewma.values, label="Smooth EWMA", color="#e76f51", lw=1.8, alpha=0.9)
ax.set_ylim(0, 3); ax.set_title(f"Athlete {aid}: ACWR — Classic vs EWMA")
ax.set_ylabel("AC ratio"); ax.legend(loc="upper left"); plt.tight_layout(); plt.show()

In [ ]:
# How closely do the two agree, and where do they diverge?
cmp = pd.DataFrame({"rolling": roll["acwr_rolling"], "ewma": ewma}).dropna()
corr = cmp["rolling"].corr(cmp["ewma"])
print(f"Correlation (rolling vs ewma): {corr:.3f}")
print(f"Mean |difference|:            {(cmp['rolling'] - cmp['ewma']).abs().mean():.3f}")
print("EWMA reacts faster: on the biggest single-day jumps it moves more than rolling.")
sns.jointplot(data=cmp, x="rolling", y="ewma", kind="reg", height=4.5, color="#457b9d")
plt.show()

## 7. Monotony & strain (Foster 1998)

Two athletes can have the same weekly load but very different injury risk. **Monotony** =
weekly mean daily load ÷ its standard deviation — high monotony means "same thing every day".
**Strain** = weekly load × monotony. Both are extra inputs to the app's injury-risk gauge.

In [ ]:
def monotony_strain(daily):
    week = daily.rolling(7, min_periods=1)
    mean = week.mean(); std = week.std(ddof=0)
    monotony = (mean / std.replace(0, np.nan)).clip(upper=5).fillna(0)
    strain = daily.rolling(7, min_periods=1).sum() * monotony
    return monotony, strain

mono, strain = monotony_strain(daily)
plt.figure(figsize=(13, 3.2))
plt.plot(mono.index, mono, label="monotony", color="#8e44ad")
plt.axhline(2.0, color="crimson", ls="--", lw=1, label="risky > 2")
plt.title(f"Athlete {aid}: training monotony"); plt.legend(); plt.tight_layout(); plt.show()

## 8. Classification (Task 2) — will an injury happen in the next 7 days?

We build a per-day feature matrix from everything above and label each day **1** if an injury
starts within the next 7 days. This is the model behind an early-warning flag. We compare Logistic
Regression and a Random Forest with the full classification metric suite + ROC-AUC.

In [ ]:
def build_features(df):
    rows = []
    for aid, g in df.groupby("athlete"):
        g = g.sort_values("date").reset_index(drop=True)
        exp = int(g["experience"].iloc[0])
        daily = daily_series(g)
        roll = rolling_acwr(daily, exp)
        ewma = ewma_acwr(daily, exp)
        mono, strain = monotony_strain(daily)
        feat = pd.DataFrame({
            "date": daily.index,
            "athlete": aid, "experience": exp,
            "daily_load": daily.values,
            "acute": roll["acute"].values,
            "chronic": roll["chronic"].values,
            "acwr_rolling": roll["acwr_rolling"].values,
            "acwr_ewma": ewma.values,
            "monotony": mono.values,
            "strain": strain.values,
        })
        # label: injury onset within the NEXT 7 days
        inj = g.set_index("date")["injury"].reindex(daily.index, fill_value=0)
        fwd = inj[::-1].rolling(7, min_periods=1).sum()[::-1]  # forward-looking sum
        feat["injury_next7"] = (fwd.values > 0).astype(int)
        rows.append(feat)
    out = pd.concat(rows, ignore_index=True).dropna()
    return out

feat = build_features(df)
print("feature rows:", len(feat), "| positive rate:", round(feat["injury_next7"].mean(), 3))
feat.head()

In [ ]:
FEATURES = ["daily_load", "acute", "chronic", "acwr_rolling", "acwr_ewma",
            "monotony", "strain", "experience"]
X = feat[FEATURES].to_numpy()
y = feat["injury_next7"].to_numpy()

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler().fit(X_tr)
X_tr_s, X_te_s = scaler.transform(X_tr), scaler.transform(X_te)

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(n_estimators=250, class_weight="balanced", random_state=42),
}
results = {}
for name, clf in models.items():
    Xt = X_tr_s if name == "LogisticRegression" else X_tr
    Xe = X_te_s if name == "LogisticRegression" else X_te
    clf.fit(Xt, y_tr)
    proba = clf.predict_proba(Xe)[:, 1]
    pred = (proba >= 0.5).astype(int)
    results[name] = dict(
        clf=clf, proba=proba, pred=pred,
        acc=accuracy_score(y_te, pred),
        prec=precision_score(y_te, pred, zero_division=0),
        rec=recall_score(y_te, pred, zero_division=0),
        f1=f1_score(y_te, pred, zero_division=0),
        auc=roc_auc_score(y_te, proba),
    )
    print(f"{name:20s} acc={results[name]['acc']:.3f} prec={results[name]['prec']:.3f} "
          f"rec={results[name]['rec']:.3f} f1={results[name]['f1']:.3f} auc={results[name]['auc']:.3f}")

In [ ]:
best = max(results, key=lambda k: results[k]["auc"])
print(f"Best by AUC: {best}\n")
print(classification_report(y_te, results[best]["pred"], zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
cm = confusion_matrix(y_te, results[best]["pred"])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["no inj", "injury"], yticklabels=["no inj", "injury"])
axes[0].set_title(f"{best} — confusion matrix"); axes[0].set_xlabel("predicted"); axes[0].set_ylabel("actual")

for name in results:
    fpr, tpr, _ = roc_curve(y_te, results[name]["proba"])
    axes[1].plot(fpr, tpr, label=f"{name} (AUC={results[name]['auc']:.2f})")
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_title("ROC curves"); axes[1].set_xlabel("false positive rate")
axes[1].set_ylabel("true positive rate"); axes[1].legend(loc="lower right")
plt.tight_layout(); plt.show()

In [ ]:
# Which features drive the risk? (Random Forest importances)
rf = results["RandomForest"]["clf"]
imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()
plt.figure(figsize=(8, 3.5))
imp.plot(kind="barh", color="#2a9d8f")
plt.title("Injury-risk feature importance (RandomForest)"); plt.tight_layout(); plt.show()
imp.sort_values(ascending=False)

## 7b. Regression (Task 1) — projecting next week's load

The coach *forecast* screen fits a trend on the month's completed weeks. As a compact demonstration
of Task 1, we predict **next week's total load** from this week's features with Linear Regression and
report MAE / MSE / RMSE + a residual plot.

In [ ]:
# Weekly aggregation per athlete, then predict week t+1 load from week t features.
wk = (df.assign(week=((df["date"] - START).dt.days // 7))
        .groupby(["athlete", "week"])
        .agg(load=("load", "sum"), sessions=("load", lambda s: (s > 0).sum()),
             rpe=("rpe", "mean"), exp=("experience", "first"))
        .reset_index())
wk["next_load"] = wk.groupby("athlete")["load"].shift(-1)
wk = wk.dropna()

Xr = wk[["load", "sessions", "rpe", "exp"]].to_numpy()
yr = wk["next_load"].to_numpy()
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, yr, test_size=0.25, random_state=42)
reg = LinearRegression().fit(Xr_tr, yr_tr)
pred = reg.predict(Xr_te)

mae = mean_absolute_error(yr_te, pred)
rmse = float(np.sqrt(mean_squared_error(yr_te, pred)))
print(f"MAE={mae:.1f}  MSE={mean_squared_error(yr_te, pred):.0f}  RMSE={rmse:.1f}  R^2={r2_score(yr_te, pred):.3f}")

plt.figure(figsize=(6, 4))
plt.scatter(pred, yr_te - pred, alpha=0.6, color="#457b9d")
plt.axhline(0, color="crimson", ls="--")
plt.xlabel("predicted next-week load"); plt.ylabel("residual"); plt.title("Residuals")
plt.tight_layout(); plt.show()

## 9. Clustering — athlete archetypes (KMeans)

Summarise each athlete by average load, monotony and share of hard days, then group them. This is the
same unsupervised step the course covers; it could power squad-level coaching views.

In [ ]:
prof = (feat.groupby("athlete")
        .agg(avg_load=("daily_load", "mean"),
             avg_acwr=("acwr_rolling", "mean"),
             avg_monotony=("monotony", "mean"),
             pct_hard=("daily_load", lambda s: float((s > s.quantile(0.75)).mean())))
        .reset_index())
Xc = StandardScaler().fit_transform(prof[["avg_load", "avg_acwr", "avg_monotony", "pct_hard"]])
k = 3
prof["cluster"] = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(Xc)

sns.scatterplot(data=prof, x="avg_load", y="avg_monotony", hue="cluster",
                size="avg_acwr", palette="Set2", sizes=(40, 200))
plt.title(f"Athlete archetypes (KMeans, k={k})"); plt.tight_layout(); plt.show()
prof.sort_values("cluster")

## 10. Export the model

Persist the best injury-risk classifier + its scaler with `joblib`, the same mechanism the live
service uses to load `models/*.pkl`. We save under a distinct name so we don't clobber the service's
production pickles.

In [ ]:
import os
os.makedirs("../models", exist_ok=True) if os.path.basename(os.getcwd()) == "notebook" else os.makedirs("models", exist_ok=True)
out_dir = "../models" if os.path.basename(os.getcwd()) == "notebook" else "models"
bundle = {"model": results[best]["clf"], "scaler": scaler, "features": FEATURES,
          "needs_scaling": best == "LogisticRegression", "auc": results[best]["auc"]}
path = os.path.join(out_dir, "acwr_injury_classifier_demo.pkl")
joblib.dump(bundle, path)
print("Saved:", path, "| AUC", round(results[best]["auc"], 3))

## 11. Conclusion

- We rebuilt TrainWise's **two ACWR methods** from first principles — Classic rolling (Gabbett) and
  bias-corrected EWMA (Williams) — with the same cold-start / covered-days guards the app uses. A
  parity test confirms these formulas match `ml/features.py` and `utils/loadSeries.js` to floating
  point, so the notebook, the ML service and the app all agree.
- **Regression** projects next-week load (Task 1); **classification** flags an injury in the next 7
  days with ROC-AUC well above chance (Task 2); **KMeans** groups athletes into training archetypes.
- Feature importance confirms the sports-science intuition: **ACWR and strain** are the strongest
  injury-risk signals — which is exactly why they drive the app's Load tab and injury-risk gauge.

**In the app:** these series are served by `GET /api/ml/trainee/<id>/analytics` and rendered on the
trainee *Load* tab and the coach *Analytics* screen with a **Classic / Smooth (EWMA)** toggle.
